In [14]:
import pandas as pd
product_meta = pd.read_csv("../12_22/final_product_with_brandtone_meta.csv")
print(list(product_meta.columns))

['brand', '상품명', 'category', 'subcategory']


In [20]:
import numpy as np
import pandas as pd
import re

# ==============================
# PATH
# ==============================
PERSONA_NPY  = "../12_22/persona_vectors.npy"
PERSONA_META = "../12_22/persona_meta.csv"

PRODUCT_NPY  = "../12_22/final_product_with_brandtone.npy"
PRODUCT_META = "../12_22/final_product_with_brandtone_meta.csv"

AMORE_FINAL  = "../12_22/amore_final.csv"
OUT_CSV      = "../12_22/persona_product_similarity_topN.csv"

TOP_N = 30
EPS = 1e-8

# ==============================
# LOAD
# ==============================
P = np.load(PERSONA_NPY).astype(np.float32)      # (P, D)
X = np.load(PRODUCT_NPY).astype(np.float32)      # (N, D)

persona_meta = pd.read_csv(PERSONA_META)
product_meta = pd.read_csv(PRODUCT_META)
amore_df     = pd.read_csv(AMORE_FINAL)

assert P.shape[0] == len(persona_meta)
assert X.shape[0] == len(product_meta)

# ==============================
# PERSONA / PRODUCT TONE SPLIT
# ==============================
RISK_DIM = 4
persona_dim = P.shape[1]
tone_dim = (persona_dim - RISK_DIM) // 2

p_tone = P[:, :tone_dim]          # (P, D)
x_brandtone = X[:, -tone_dim:]    # (N, D)

def normalize(v):
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + EPS)

sim_brand = normalize(p_tone) @ normalize(x_brandtone).T   # (P, N)

# ==============================
# PRODUCT META JOIN (ROW ID 유지)
# ==============================
product_meta = product_meta.reset_index(drop=True)
product_meta["__row_id"] = product_meta.index   # 🔴 기준축 (X index)

amore_df["brand"] = amore_df["brand"].astype(str)
amore_df["상품명"] = amore_df["상품명"].astype(str)
product_meta["brand"] = product_meta["brand"].astype(str)
product_meta["상품명"] = product_meta["상품명"].astype(str)

product_meta = product_meta.merge(
    amore_df[["brand", "상품명", "전성분"]],
    on=["brand", "상품명"],
    how="left"
)

# ==============================
# INGREDIENT NORMALIZE
# ==============================
def norm(s):
    s = str(s).lower()
    s = re.sub(r"[\*\#\(\)\[\]\{\}\/\:\.]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

product_meta["_ing_norm"] = product_meta["전성분"].fillna("").map(norm)

# ==============================
# PERSONA GLOB RULE
# ==============================
PERSONA_GLOB = {
    "persona_1": ["하이알루로", "히알루로", "hyalur", "hyaluron",
                  "시카", "centella", "asiatic", "madecass", "병풀"],
    "persona_2": ["나이아신", "niacin", "niacinamide"],
    "persona_3": ["병풀", "centella", "asiatica", "인삼", "ginseng", "panax"],
}

def glob_score(patterns, text):
    hits = sum(1 for p in patterns if p in text)
    if hits == 0:
        return 0.0
    return min(1.0, hits / len(patterns))

# ==============================
# WEIGHTS
# ==============================
persona_weights = {
    "persona_1": dict(brand=0.6, ing=0.4),
    "persona_2": dict(brand=0.5, ing=0.5),
    "persona_3": dict(brand=0.4, ing=0.6),
}

# ==============================
# SIMILARITY + TOP-N
# ==============================
rows = []
product_cols = [c for c in product_meta.columns if not c.startswith("_")]

N = X.shape[0]

for p_idx in range(P.shape[0]):
    pid = persona_meta.loc[p_idx, "persona_id"]
    patterns = [norm(p) for p in PERSONA_GLOB.get(pid, [])]
    w = persona_weights.get(pid, dict(brand=0.6, ing=0.4))

    # 🔴 핵심: X 기준 길이로 ingredient score 생성
    sim_ing_full = np.zeros(N, dtype=np.float32)

    tmp_scores = product_meta["_ing_norm"].apply(
        lambda txt: glob_score(patterns, txt)
    ).values

    sim_ing_full[product_meta["__row_id"].values] = tmp_scores

    sim = (w["brand"] * sim_brand[p_idx]) + (w["ing"] * sim_ing_full)

    top_idx = np.argpartition(sim, -TOP_N)[-TOP_N:]
    top_idx = top_idx[np.argsort(sim[top_idx])[::-1]]

    for rank, prod_idx in enumerate(top_idx, start=1):
        row = {
            "persona_id": pid,
            "rank": rank,
            "similarity": float(sim[prod_idx]),
            "sim_brand": float(sim_brand[p_idx, prod_idx]),
            "sim_ingredient": float(sim_ing_full[prod_idx]),
            "product_index": int(prod_idx),
        }
        for c in product_cols:
            row[c] = product_meta.loc[prod_idx, c]
        rows.append(row)

# ==============================
# SAVE
# ==============================
result_df = pd.DataFrame(rows)
result_df.to_csv(OUT_CSV, index=False)

print("✅ saved:", OUT_CSV)
display(
    result_df.sort_values(["persona_id", "rank"])
             .groupby("persona_id", as_index=False)
             .head(10)
)

✅ saved: ../12_22/persona_product_similarity_topN.csv


,persona_id,rank,similarity,sim_brand,sim_ingredient,product_index,brand,상품명,category,subcategory,전성분
0,persona_1,1,0.651584,0.863751,0.333333,337,에스쁘아,이지 블렌딩 컨실러 10g,건기식,기능성식품,"정제수, 티타늄디옥사이드 (CI 77891), 카프릴릴메티콘, 코코-카프릴레이트/카..."
1,persona_1,2,0.651584,0.863751,0.333333,313,에스쁘아,아이 코어 팔레트 기획세트,기타,미분류,"""[Minnie] 탤크, 티타늄디옥사이드 (CI 77891), 마이카, 메틸메타크릴..."
2,persona_1,3,0.638239,0.915584,0.222222,1180,이니스프리,국화 여성 청결제 200ml,기타,미분류,"정제수, 부틸렌글라이콜, 하이드록시프로필메틸셀룰로오스, 라우릴베타인, 에탄올, 국화..."
3,persona_1,4,0.638239,0.915584,0.222222,787,비레디,블루 수분 선크림 1+1 기획세트 SPF50+PA++++,선케어,선크림,"정제수, 부틸렌글라이콜, 호모살레이트, 에칠헥실살리실레이트, 변성알코올, 비스-에칠..."
4,persona_1,5,0.638239,0.915584,0.222222,789,라네즈,블루 수분 선크림 1+1 기획세트 SPF50+PA++++,선케어,선크림,"정제수, 부틸렌글라이콜, 호모살레이트, 에칠헥실살리실레이트, 변성알코올, 비스-에칠..."
5,persona_1,6,0.638239,0.915584,0.222222,783,비레디,트루톤 로션 40ml,기타,미분류,"정제수, 호모살레이트, 글리세린, 디에칠아미노하이드록시벤조일헥실벤조에이트, 에칠헥실..."
6,persona_1,7,0.638239,0.915584,0.222222,785,라네즈,트루톤 로션 40ml,기타,미분류,"정제수, 호모살레이트, 글리세린, 디에칠아미노하이드록시벤조일헥실벤조에이트, 에칠헥실..."
7,persona_1,8,0.638239,0.915584,0.222222,1222,이니스프리,하우스 스펀지 6p,기타,미분류,상세 참조
8,persona_1,9,0.638239,0.915584,0.222222,1228,이니스프리,인텐시브 롱래스팅 선스크린 EX SPF50+/PA++++ 60ml,선케어,선크림,"정제수, 징크옥사이드, 프로필헵틸카프릴레이트, 다이부틸아디페이트, 부틸렌글라이콜, ..."
9,persona_1,10,0.638239,0.915584,0.222222,1226,이니스프리,라이트 피팅 파운데이션 SPF20/PA++ 30ml,선케어,선크림,"정제수, 티타늄디옥사이드 (CI 77891), 다이부틸아디페이트, 카프릴릴메티콘, ..."


In [6]:
import pandas as pd

ingredient_meta = pd.read_csv("../12_22/ingredient_meta.csv")

# 1) 전체 성분 수
print("총 성분 수:", len(ingredient_meta))

# 2) 상위 50개 성분 샘플
ingredient_meta["ingredient_name"].head(50)

총 성분 수: 3149


0                                  #1. 부메랑 칼슘티타늄보로실리케이트
1                                          #2. 피치파우트 탤크
2                                               #핑키 마이카
3                                              (1제) 정제수
4                                          (STEP 1) 정제수
5                            (로지) 하이드로제네이티드폴리(C6-14올레핀)
6                                             (소프너) 정제수
7                         (스카이코랄) 하이드로제네이티드폴리(C6-14올레핀)
8                  *BLACK TEA PEPTIDE ACTIVATORTM / 정제수
9                                       *맨 리차징 토너 : 정제수
10                                   *소듐아세틸레이티드하이알루로네이트
11                                          *소듐하이알루로네이트
12                                    *소듐하이알루로네이트크로스폴리머
13                                         *포타슘하이알루로네이트
14                               *하이드록시프로필트라이모늄하이알루로네이트
15                                  *하이드롤라이즈드소듐하이알루로네이트
16                                   *하이드롤라이즈드하이알루로닉애씨드
17                                           *하이

In [7]:
# 3) 특정 키워드 포함 성분이 실제로 있는지
keywords = ["centella", "asiatica", "hyal", "niacin", "ginseng"]

for kw in keywords:
    matched = ingredient_meta[
        ingredient_meta["ingredient_name"].str.lower().str.contains(kw, na=False)
    ]
    print(f"\n[{kw}] 매칭 수:", len(matched))
    print(matched["ingredient_name"].head(10).tolist())


[centella] 매칭 수: 0
[]

[asiatica] 매칭 수: 0
[]

[hyal] 매칭 수: 0
[]

[niacin] 매칭 수: 0
[]

[ginseng] 매칭 수: 0
[]
